# Interstate Border Settlement Analysis in Nigeria
## Seven-Chapter Google Colab Notebook

This notebook identifies settlements located within **5 km on both sides of a shared State/FCT boundary** in Nigeria.

The user selects two States/FCTs. The notebook verifies that they share a border, constructs a 5 km corridor inside each State/FCT, extracts GRID3 settlement extents and names, adds roads, bridges, drainage and health facilities, and produces professional border-focused maps and CSV tables.

### Main outputs
- `interstate_border_settlements_5km.csv`
- `interstate_border_summary.csv`
- `01_interstate_border_multipanel_map.png`
- `02_interstate_border_settlements_map.png`
- ZIP containing the CSV and PNG outputs

### Important interpretation
The 5 km distance is measured in a projected coordinate system from the shared administrative border. It is a geographic screening distance, not travel time. Settlement extents are GRID3 mapped extents and are not official community administrative boundaries.


# Chapter 1 — Prepare the interstate-border analysis environment
Use the same validated administrative, GRID3, OpenStreetMap and health-facility sources already registered in the course GitHub repository.


In [ ]:
# Cell 1 — Install tools, mount Drive, clone the course repository, and load State/FCT lookup
!pip -q install geopandas pyogrio shapely pyproj requests ipywidgets matplotlib mapclassify adjustText osm2geojson

from pathlib import Path
import json, re, subprocess, sys, requests, hashlib, zipfile, shutil
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")

REPO = "zubairgis/nigeria-population-demography-course"
RAW = f"https://raw.githubusercontent.com/{REPO}/main"

TEMP_ROOT = Path("/content/interstate_border_course")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/Nigeria_Interstate_Border_Settlements")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = TEMP_ROOT / "repo"
if REPO_DIR.exists():
    subprocess.run(["git","-C",str(REPO_DIR),"pull","-q"], check=False)
else:
    subprocess.run(["git","clone","-q",f"https://github.com/{REPO}.git",str(REPO_DIR)], check=True)

states = pd.read_csv(
    REPO_DIR/"data/lookups/admin_states_source.csv",
    dtype=str
)

assert len(states) == 37

safe = lambda s: re.sub(r"[^A-Za-z0-9]+","_",str(s)).strip("_")

print("Repository ready:", REPO_DIR)
print("State/FCT choices:", len(states))
print("Persistent output root:", DRIVE_ROOT)


# Chapter 2 — Select two States/FCTs
Choose the two administrative units whose **shared interstate boundary** you want to study. The notebook will stop if the selected pair does not share a border.


In [ ]:
# Cell 2 — Select two States/FCTs
state_options = sorted(states["adm1nm"].astype(str).unique())

_default_1 = "Bayelsa" if "Bayelsa" in state_options else state_options[0]
_default_2 = "Delta" if "Delta" in state_options else state_options[1]

state1_dropdown = widgets.Dropdown(
    options=state_options,
    value=_default_1,
    description="State 1:"
)

state2_dropdown = widgets.Dropdown(
    options=state_options,
    value=_default_2,
    description="State 2:"
)

display(widgets.VBox([state1_dropdown, state2_dropdown]))

print("Choose two neighbouring States/FCTs, then run Cell 3.")


# Chapter 3 — Define the shared border and the 5 km corridor
The shared border is derived from the SALB administrative geometry already used by the main course notebook.

A **5 km inward corridor** is created inside each selected State/FCT. Together these two zones represent settlements located within 5 km on either side of the interstate boundary.


In [ ]:
# Cell 3 — Load the two State/FCT boundaries, verify adjacency, and build the 5 km corridor
STATE1 = state1_dropdown.value
STATE2 = state2_dropdown.value

if STATE1 == STATE2:
    raise ValueError("Choose two different States/FCTs.")

STATE1_ID = states.loc[states.adm1nm.eq(STATE1),"adm1cd"].iloc[0]
STATE2_ID = states.loc[states.adm1nm.eq(STATE2),"adm1cd"].iloc[0]

admin_root = (
    REPO_DIR /
    "data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state"
)

state1_lgas = gpd.read_file(
    admin_root/f"{STATE1_ID}_{safe(STATE1)}.geojson"
).to_crs(4326)

state2_lgas = gpd.read_file(
    admin_root/f"{STATE2_ID}_{safe(STATE2)}.geojson"
).to_crs(4326)

state1 = state1_lgas.dissolve(by="adm1cd", as_index=False)
state2 = state2_lgas.dissolve(by="adm1cd", as_index=False)

pair = gpd.GeoDataFrame(
    pd.concat([
        state1.assign(state_name=STATE1, state_code=STATE1_ID),
        state2.assign(state_name=STATE2, state_code=STATE2_ID)
    ], ignore_index=True),
    geometry="geometry",
    crs=4326
)

PAIR_CRS = pair.estimate_utm_crs()
if PAIR_CRS is None:
    raise ValueError("Could not determine a projected CRS for the selected State pair.")

s1p = state1.to_crs(PAIR_CRS).geometry.iloc[0]
s2p = state2.to_crs(PAIR_CRS).geometry.iloc[0]

state_distance_m = float(s1p.distance(s2p))

# Exact common boundary from the same SALB source.
shared_border = s1p.boundary.intersection(s2p.boundary)

def _line_only(geom):
    if geom.is_empty:
        return geom
    if geom.geom_type in ["LineString","MultiLineString"]:
        return geom
    if geom.geom_type == "GeometryCollection":
        lines = [
            g for g in geom.geoms
            if g.geom_type in ["LineString","MultiLineString"]
        ]
        if not lines:
            return geom
        return gpd.GeoSeries(lines, crs=PAIR_CRS).union_all()
    return geom

shared_border = _line_only(shared_border)

# Small tolerance fallback for tiny coordinate discrepancies.
if shared_border.is_empty or getattr(shared_border, "length", 0) < 100:
    if state_distance_m > 150:
        raise ValueError(
            f"{STATE1} and {STATE2} do not appear to share a border "
            f"(minimum separation about {state_distance_m/1000:.2f} km)."
        )
    shared_border = _line_only(
        s1p.boundary.intersection(s2p.buffer(75))
    )

if shared_border.is_empty or getattr(shared_border, "length", 0) < 100:
    raise ValueError(
        "A usable shared border line could not be derived. "
        "Choose two States/FCTs that share a land boundary."
    )

BUFFER_M = 5000.0

corridor1_geom = shared_border.buffer(BUFFER_M).intersection(s1p)
corridor2_geom = shared_border.buffer(BUFFER_M).intersection(s2p)
corridor_union_geom = corridor1_geom.union(corridor2_geom)

border_gdf_p = gpd.GeoDataFrame(
    {"border":[f"{STATE1}–{STATE2}"]},
    geometry=[shared_border],
    crs=PAIR_CRS
)

corridors_p = gpd.GeoDataFrame(
    {
        "state":[STATE1,STATE2],
        "state_code":[STATE1_ID,STATE2_ID],
        "side":[1,2]
    },
    geometry=[corridor1_geom,corridor2_geom],
    crs=PAIR_CRS
)

corridors = corridors_p.to_crs(4326)
border_gdf = border_gdf_p.to_crs(4326)

BBOX = tuple(float(x) for x in corridors.total_bounds)

STUDENT_WORK_DIR = (
    DRIVE_ROOT /
    f"{safe(STATE1)}__{safe(STATE2)}"
)
STUDENT_WORK_DIR.mkdir(parents=True, exist_ok=True)

WORK = (
    TEMP_ROOT /
    f"{safe(STATE1)}__{safe(STATE2)}"
)
WORK.mkdir(parents=True, exist_ok=True)

print("State pair:", STATE1, "and", STATE2)
print("Shared border length:", f"{shared_border.length/1000:,.2f} km")
print("Corridor rule: 5 km inside each State/FCT")
print("State 1 corridor area:", f"{corridor1_geom.area/1e6:,.2f} km²")
print("State 2 corridor area:", f"{corridor2_geom.area/1e6:,.2f} km²")
print("Output folder:", STUDENT_WORK_DIR)


# Chapter 4 — Identify settlements within 5 km of the interstate border
GRID3 Nigeria Settlement Extents v3.1 are read from the same GitHub release already registered in the repository. GRID3 settlement-name points are queried using the repository's validated source script.

Settlements are clipped to the appropriate side of the interstate corridor and assigned to the State/FCT and LGA in which the component lies.


In [ ]:
# Cell 4 — Load GRID3 settlement extents and names, then build border-corridor settlement components
CACHE = DRIVE_ROOT/"source_cache"
CACHE.mkdir(parents=True, exist_ok=True)

zip_path = CACHE/"GRID3_NGA_settlement_extents_v3_1_gpkg.zip"
gpkg_path = CACHE/"GRID3_NGA_settlement_extents_v3_1.gpkg"

if not gpkg_path.exists():
    if not zip_path.exists():
        source_cfg = json.loads(
            (REPO_DIR/"config/source_endpoints.json").read_text()
        )
        url = source_cfg["grid3_settlement_extents_v3_1"]["download_url"]
        print("Downloading GRID3 settlement extents once to Drive cache...")
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(zip_path,"wb") as out:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        out.write(chunk)

    with zipfile.ZipFile(zip_path) as z:
        member = [
            n for n in z.namelist()
            if Path(n).name == "GRID3_NGA_settlement_extents_v3_1.gpkg"
        ][0]
        with z.open(member) as src, open(gpkg_path,"wb") as dst:
            shutil.copyfileobj(src,dst)

print("Settlement source:", gpkg_path)

ext = gpd.read_file(
    gpkg_path,
    bbox=BBOX
).to_crs(4326)

required = {"building_count","building_area","type","geometry"}
missing = required - set(ext.columns)
if missing:
    raise ValueError(f"Unexpected GRID3 v3.1 schema; missing: {sorted(missing)}")

# Stable source ID from geometry.
def stable_geom_id(g):
    return "v31_" + hashlib.sha1(g.wkb).hexdigest()[:20]

ext = ext.copy()
ext["source_settlement_id"] = ext.geometry.map(stable_geom_id)
ext["source_building_count"] = pd.to_numeric(
    ext["building_count"], errors="coerce"
)

# Clip the same source settlement layer separately to each side.
components = []
for side_row in corridors.itertuples():
    side_geom = gpd.GeoDataFrame(
        {"state":[side_row.state],"state_code":[side_row.state_code]},
        geometry=[side_row.geometry],
        crs=4326
    )

    comp = gpd.overlay(
        ext,
        side_geom,
        how="intersection",
        keep_geom_type=True
    )

    if len(comp):
        comp["corridor_state"] = side_row.state
        comp["corridor_state_code"] = side_row.state_code
        components.append(comp)

if not components:
    raise ValueError("No GRID3 settlement extents were found in the 5 km border corridor.")

settlements = gpd.GeoDataFrame(
    pd.concat(components, ignore_index=True),
    geometry="geometry",
    crs=4326
)

# Projected geometry metrics.
sp = settlements.to_crs(PAIR_CRS)
source_area_lookup = (
    ext.to_crs(PAIR_CRS)
    .assign(_source_area=lambda d:d.geometry.area)
    .set_index("source_settlement_id")["_source_area"]
    .to_dict()
)

settlements["settlement_area_km2"] = sp.geometry.area.to_numpy()/1e6
settlements["component_fraction_of_source"] = [
    min(1.0, max(0.0, a/source_area_lookup.get(sid,a)))
    for a,sid in zip(sp.geometry.area,settlements["source_settlement_id"])
]
settlements["estimated_component_buildings"] = (
    settlements["source_building_count"] *
    settlements["component_fraction_of_source"]
)

# Distance from each settlement representative point to the shared border.
rep_pts_p = sp.geometry.representative_point()
settlements["distance_to_shared_border_km"] = (
    rep_pts_p.distance(shared_border).to_numpy()/1000.0
)

# Assign LGA by representative point.
all_pair_lgas = gpd.GeoDataFrame(
    pd.concat([state1_lgas,state2_lgas],ignore_index=True),
    geometry="geometry",
    crs=4326
)

rep_wgs = gpd.GeoDataFrame(
    settlements[["source_settlement_id","corridor_state"]].copy(),
    geometry=gpd.GeoSeries(rep_pts_p,crs=PAIR_CRS).to_crs(4326),
    crs=4326
)

lga_join = gpd.sjoin(
    rep_wgs,
    all_pair_lgas[["adm1cd","adm1nm","adm2cd","adm2nm","geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"],errors="ignore")

settlements["lga_code"] = lga_join["adm2cd"].to_numpy()
settlements["lga_name"] = lga_join["adm2nm"].to_numpy()

# Download GRID3 settlement-name points only for the corridor bbox using repository script.
DL = WORK/"downloads"
DL.mkdir(exist_ok=True)

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","settlement_names",
    "--output",str(DL),
    "--bbox",*[str(x) for x in BBOX]
], check=True)

names = gpd.read_file(
    DL/"settlement_names.geojson"
).to_crs(4326)

names = gpd.clip(
    names,
    gpd.GeoDataFrame(geometry=[corridors.geometry.union_all()],crs=4326)
)

# Match names to settlement components.
if len(names):
    nj = gpd.sjoin(
        names,
        settlements[["geometry"]],
        how="inner",
        predicate="within"
    )

    name_field = "set_name" if "set_name" in nj.columns else (
        "name" if "name" in nj.columns else None
    )

    if name_field:
        name_summary = (
            nj.groupby("index_right")[name_field]
            .apply(lambda x:" | ".join(sorted({
                str(v).strip() for v in x.dropna()
                if str(v).strip()
            })))
        )
        settlements["settlement_name"] = settlements.index.map(name_summary)
    else:
        settlements["settlement_name"] = None
else:
    settlements["settlement_name"] = None

# Stable border-analysis serials.
sort_tmp = settlements.to_crs(PAIR_CRS).copy()
rp = sort_tmp.geometry.representative_point()
order = pd.DataFrame({
    "idx":settlements.index,
    "north":rp.y,
    "east":rp.x
}).sort_values(["north","east"],ascending=[False,True])

serial_map = {
    idx:n for n,idx in enumerate(order["idx"],start=1)
}
settlements["settlement_serial"] = [
    serial_map[i] for i in settlements.index
]

print("Settlement components in 5 km + 5 km corridor:",len(settlements))
display(
    settlements[[
        "settlement_serial","corridor_state","lga_name",
        "settlement_name","type","source_building_count",
        "distance_to_shared_border_km"
    ]].head(30)
)


# Chapter 5 — Add roads, bridges, drainage and health facilities
The analysis uses OpenStreetMap roads/bridges, the repository's cached drainage layers, and the same GRID3/NHFR health-facility source used in the main course notebook.


In [ ]:
# Cell 5 — Load roads, bridges, drainage, and health facilities for the 10 km-wide interstate corridor
import osm2geojson

corridor_union_wgs = gpd.GeoDataFrame(
    geometry=[corridors.geometry.union_all()],
    crs=4326
)

minx,miny,maxx,maxy = BBOX

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter"
]

overpass_query = f"""
[out:json][timeout:180];
(
  way["highway"]({miny},{minx},{maxy},{maxx});
);
out geom;
"""

osm_json = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        rr = requests.post(
            endpoint,
            data={"data":overpass_query},
            timeout=210,
            headers={"User-Agent":"Hensard-University-Interstate-Border-Course/1.0"}
        )
        rr.raise_for_status()
        osm_json = rr.json()
        print("OSM roads source:",endpoint)
        break
    except Exception as e:
        print("Overpass unavailable:",endpoint,type(e).__name__)

if osm_json is None:
    OSM_ROADS = gpd.GeoDataFrame(
        columns=["highway","name","bridge","geometry"],
        geometry="geometry",crs=4326
    )
else:
    gj = osm2geojson.json2geojson(osm_json)
    OSM_ROADS = gpd.GeoDataFrame.from_features(
        gj["features"],crs=4326
    )

    if len(OSM_ROADS):
        OSM_ROADS = OSM_ROADS[
            OSM_ROADS.geometry.geom_type.isin(["LineString","MultiLineString"])
        ].copy()

        if "tags" in OSM_ROADS.columns:
            def _tag(v,key):
                return v.get(key) if isinstance(v,dict) else None
            for key in ["highway","name","bridge","layer"]:
                if key not in OSM_ROADS.columns:
                    OSM_ROADS[key] = OSM_ROADS["tags"].apply(
                        lambda v,k=key:_tag(v,k)
                    )

        for key in ["highway","name","bridge","layer"]:
            if key not in OSM_ROADS.columns:
                OSM_ROADS[key] = None

        OSM_ROADS = OSM_ROADS[
            OSM_ROADS["highway"].notna()
        ][["highway","name","bridge","layer","geometry"]].copy()

        OSM_ROADS = gpd.clip(
            OSM_ROADS,
            corridor_union_wgs
        )

if len(OSM_ROADS) and "bridge" in OSM_ROADS.columns:
    bv = OSM_ROADS["bridge"].astype(str).str.lower()
    OSM_BRIDGES = OSM_ROADS[
        OSM_ROADS["bridge"].notna() &
        ~bv.isin(["","no","false","0","nan","none"])
    ].copy()
else:
    OSM_BRIDGES = gpd.GeoDataFrame(
        columns=["highway","name","bridge","geometry"],
        geometry="geometry",crs=4326
    )

# Drainage: select repository files from manifest by State code.
drain_manifest = json.loads(
    (REPO_DIR/"data/drainage/manifest.json").read_text()
)

drain_parts = []
for state_code in [STATE1_ID,STATE2_ID]:
    rec = next(
        x for x in drain_manifest["states"]
        if x["adm1cd"] == state_code
    )
    d = gpd.read_file(
        REPO_DIR/"data/drainage"/rec["file"]
    ).to_crs(4326)
    drain_parts.append(d)

DRAINAGE = gpd.GeoDataFrame(
    pd.concat(drain_parts,ignore_index=True),
    geometry="geometry",
    crs=4326
)
DRAINAGE = gpd.clip(DRAINAGE,corridor_union_wgs)

# Health facilities from the same registered source workflow.
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","health_v2",
    "--output",str(DL),
    "--bbox",*[str(x) for x in BBOX]
], check=True)

health = gpd.read_file(
    DL/"health_v2.geojson"
).to_crs(4326)

health = gpd.clip(
    health,
    corridor_union_wgs
)

# Normalize a usable health facility name field.
health_name_candidates = [
    "facility_name","name","facility_n","facilityna","health_fac"
]
health_name_field = next(
    (c for c in health_name_candidates if c in health.columns),
    None
)

if health_name_field and health_name_field != "facility_name":
    health["facility_name"] = health[health_name_field].astype(str)
elif "facility_name" not in health.columns:
    health["facility_name"] = "Health facility"

print("Road features:",len(OSM_ROADS))
print("Bridge features:",len(OSM_BRIDGES))
print("Drainage features:",len(DRAINAGE))
print("Health facilities:",len(health))


# Chapter 6 — Build the interstate settlement analysis table
The CSV is designed for Excel and development planning. Each row represents a settlement component on one side of the shared border.

Important variables include State/FCT, LGA, settlement type, building count, area, distance to the interstate border, and distance to the nearest mapped health facility.


In [ ]:
# Cell 6 — Calculate nearest-health distance and prepare the final CSV tables
sett_p = settlements.to_crs(PAIR_CRS).copy()
sett_points = sett_p.geometry.representative_point()

health_p = health.to_crs(PAIR_CRS).copy() if len(health) else health.copy()

nearest_health_name = []
nearest_health_km = []

for p in sett_points:
    if len(health_p):
        dist = health_p.geometry.distance(p)
        idx = dist.idxmin()
        nearest_health_km.append(float(dist.loc[idx])/1000.0)
        nearest_health_name.append(
            str(health_p.loc[idx].get("facility_name","Health facility"))
        )
    else:
        nearest_health_km.append(np.nan)
        nearest_health_name.append(np.nan)

settlements["nearest_health_facility"] = nearest_health_name
settlements["nearest_health_distance_km"] = nearest_health_km

rep_wgs = gpd.GeoSeries(
    sett_points,
    crs=PAIR_CRS
).to_crs(4326)

settlements["longitude"] = rep_wgs.x.to_numpy()
settlements["latitude"] = rep_wgs.y.to_numpy()
settlements["within_5km_shared_border"] = True
settlements["border_pair"] = f"{STATE1}–{STATE2}"

final_cols = [
    "settlement_serial",
    "border_pair",
    "corridor_state",
    "corridor_state_code",
    "lga_name",
    "lga_code",
    "settlement_name",
    "type",
    "source_settlement_id",
    "source_building_count",
    "estimated_component_buildings",
    "component_fraction_of_source",
    "settlement_area_km2",
    "distance_to_shared_border_km",
    "nearest_health_facility",
    "nearest_health_distance_km",
    "longitude",
    "latitude",
    "within_5km_shared_border"
]

settlement_csv = settlements[
    [c for c in final_cols if c in settlements.columns]
].copy().sort_values(
    ["corridor_state","distance_to_shared_border_km","settlement_serial"]
)

summary_csv = (
    settlement_csv.groupby("corridor_state",dropna=False)
    .agg(
        settlement_components=("settlement_serial","count"),
        source_buildings=("source_building_count","sum"),
        estimated_component_buildings=("estimated_component_buildings","sum"),
        mean_border_distance_km=("distance_to_shared_border_km","mean"),
        median_border_distance_km=("distance_to_shared_border_km","median"),
        mean_health_distance_km=("nearest_health_distance_km","mean")
    )
    .reset_index()
)

display(settlement_csv.head(40))
display(summary_csv.round(2))


# Chapter 7 — Produce professional border-focused maps and export CSV/PNG outputs
Panels A–C are zoomed to the shared interstate border corridor. Panel D provides the two-State locator and analysis summary.

The final ZIP contains only CSV and PNG outputs.


In [ ]:
# Cell 7 — Export the interstate-border CSVs and professional multi-panel maps
from adjustText import adjust_text

EXPORT = STUDENT_WORK_DIR/"Final_Results"
EXPORT.mkdir(parents=True,exist_ok=True)

for p in EXPORT.iterdir():
    if p.is_file():
        p.unlink()

settlement_csv.to_csv(
    EXPORT/"interstate_border_settlements_5km.csv",
    index=False
)
summary_csv.to_csv(
    EXPORT/"interstate_border_summary.csv",
    index=False
)

# Project all layers once for mapping.
states_pair_p = pair.to_crs(PAIR_CRS)
corridors_map_p = corridors.to_crs(PAIR_CRS)
border_map_p = border_gdf.to_crs(PAIR_CRS)
sett_map_p = settlements.to_crs(PAIR_CRS)
roads_p = OSM_ROADS.to_crs(PAIR_CRS) if len(OSM_ROADS) else OSM_ROADS.copy()
bridges_p = OSM_BRIDGES.to_crs(PAIR_CRS) if len(OSM_BRIDGES) else OSM_BRIDGES.copy()
drain_p = DRAINAGE.to_crs(PAIR_CRS) if len(DRAINAGE) else DRAINAGE.copy()
health_map_p = health.to_crs(PAIR_CRS) if len(health) else health.copy()

xmin,ymin,xmax,ymax = corridors_map_p.total_bounds
padx=max((xmax-xmin)*0.03,500)
pady=max((ymax-ymin)*0.03,500)

STATE_COLORS = {
    STATE1:"#d9edf7",
    STATE2:"#fde0dd"
}

SETT_COLORS = {
    "built-up":"#d73027",
    "built_up":"#d73027",
    "urban":"#d73027",
    "city":"#d73027",
    "town":"#fc8d59",
    "large":"#fc8d59",
    "medium":"#91bfdb",
    "village":"#fee08b",
    "small":"#c99700",
    "hamlet":"#f46d43"
}

ROAD_STYLE = {
    "motorway":("#d73027",1.0),
    "trunk":("#f46d43",0.9),
    "primary":("#fdae61",0.8),
    "secondary":("#fee08b",0.7),
    "tertiary":("#8c8c8c",0.55),
    "residential":("#bdbdbd",0.38),
    "unclassified":("#cccccc",0.34),
    "service":("#dddddd",0.28)
}

DRAIN_STYLE = {
    "river":("#08519c",0.9,"-"),
    "stream":("#4292c6",0.45,"-"),
    "canal":("#6baed6",0.45,"--"),
    "drain":("#9ecae1",0.35,":")
}

def classify_type(v):
    s=str(v).lower()
    for k in ["built-up","built_up","urban","city","town","large","medium","village","small","hamlet"]:
        if k in s:
            return k
    return "other"

def draw_roads(ax):
    if not len(roads_p):
        return
    for hw,grp in roads_p.groupby(roads_p["highway"].fillna("unclassified").astype(str)):
        c,lw=ROAD_STYLE.get(hw,("#cccccc",0.3))
        grp.plot(ax=ax,color=c,linewidth=lw,zorder=5)

def draw_drainage(ax):
    if not len(drain_p):
        return
    if "waterway" not in drain_p.columns:
        drain_p.plot(ax=ax,color="#4292c6",linewidth=0.45,zorder=4)
        return
    for w,grp in drain_p.groupby(drain_p["waterway"].fillna("stream").astype(str)):
        c,lw,ls=DRAIN_STYLE.get(w,("#4292c6",0.4,"-"))
        grp.plot(ax=ax,color=c,linewidth=lw,linestyle=ls,zorder=4)

def add_grid(ax):
    ax.grid(True,linestyle="--",linewidth=0.35,color="#b0b0b0",alpha=0.6)
    ax.tick_params(labelsize=6)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x,pos:f"{x/1000:.0f}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y,pos:f"{y/1000:.0f}"))
    ax.set_xlabel("Easting (km)",fontsize=7)
    ax.set_ylabel("Northing (km)",fontsize=7)

def add_north(ax):
    ax.annotate(
        "N",xy=(0.96,0.88),xytext=(0.96,0.79),
        xycoords="axes fraction",
        arrowprops=dict(facecolor="black",width=1.7,headwidth=6),
        ha="center",va="center",fontsize=8,fontweight="bold"
    )

def add_scale(ax):
    x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim()
    width=x1-x0; height=y1-y0
    choices=np.array([1000,2000,5000,10000,20000,50000],dtype=float)
    length=float(choices[np.argmin(np.abs(choices-width*0.14))])
    sx=x0+width*0.045; sy=y0+height*0.014
    ax.plot([sx,sx+length],[sy,sy],color="black",linewidth=2.2,zorder=30)
    ax.plot([sx,sx],[sy-height*.006,sy+height*.006],color="black",linewidth=1,zorder=30)
    ax.plot([sx+length,sx+length],[sy-height*.006,sy+height*.006],color="black",linewidth=1,zorder=30)
    ax.text(sx+length/2,sy+height*.01,f"{length/1000:g} km",ha="center",va="bottom",fontsize=6)

def apply_extent(ax):
    ax.set_xlim(xmin-padx,xmax+padx)
    ax.set_ylim(ymin-pady,ymax+pady)
    add_grid(ax); add_north(ax); add_scale(ax)

# -------------------------------------------------------
# FOUR-PANEL MAP
# -------------------------------------------------------
fig,axs=plt.subplots(2,2,figsize=(18,14))
ax1,ax2,ax3,ax4=axs.ravel()

# Panel A — interstate corridor context
for st,grp in corridors_map_p.groupby("state"):
    grp.plot(ax=ax1,facecolor=STATE_COLORS.get(st,"#eeeeee"),edgecolor="none",alpha=.72,zorder=1)

border_map_p.plot(ax=ax1,color="black",linewidth=2.0,zorder=8)
sett_map_p.plot(ax=ax1,facecolor="#7fcdbb",edgecolor="none",alpha=.8,zorder=3)
draw_drainage(ax1); draw_roads(ax1)

if len(health_map_p):
    health_map_p.plot(ax=ax1,marker="+",color="#b2182b",markersize=30,zorder=10)

ax1.set_title("A. Shared State Border and 5 km Corridor on Both Sides",fontsize=12,fontweight="bold")
apply_extent(ax1)

handles_a=[
    Patch(facecolor=STATE_COLORS[STATE1],label=f"{STATE1}: within 5 km"),
    Patch(facecolor=STATE_COLORS[STATE2],label=f"{STATE2}: within 5 km"),
    Line2D([0],[0],color="black",lw=2,label="Shared State border"),
    Patch(facecolor="#7fcdbb",label="GRID3 settlement extent"),
    Line2D([0],[0],marker="+",color="#b2182b",linestyle="None",markersize=8,label="Health facility")
]
ax1.legend(handles=handles_a,loc="lower right",fontsize=6,title="Legend",title_fontsize=7,ncol=2)

# Panel B — settlement types + numeric serials
for st,grp in corridors_map_p.groupby("state"):
    grp.plot(ax=ax2,facecolor=STATE_COLORS.get(st,"#eeeeee"),edgecolor="none",alpha=.35,zorder=1)

border_map_p.plot(ax=ax2,color="black",linewidth=1.8,zorder=8)

tmp=sett_map_p.copy()
tmp["_type_std"]=tmp["type"].apply(classify_type) if "type" in tmp.columns else "other"
used_types=[]
for typ,grp in tmp.groupby("_type_std"):
    c=SETT_COLORS.get(typ,"#bdbdbd")
    grp.plot(ax=ax2,facecolor=c,edgecolor="none",alpha=.9,zorder=3)
    used_types.append((typ,c))

rp=tmp.geometry.representative_point()
texts=[]
for i,(_,r) in enumerate(tmp.iterrows()):
    p=rp.iloc[i]
    texts.append(ax2.text(p.x,p.y,str(int(r["settlement_serial"])),fontsize=3.5,ha="center",va="center",zorder=10))
if len(texts) <= 220:
    adjust_text(texts,ax=ax2,expand=(1.05,1.12),force_text=(.25,.35),ensure_inside_axes=True,max_move=(10,10))

draw_drainage(ax2); draw_roads(ax2)
ax2.set_title("B. Settlement Types and Serial Numbers",fontsize=12,fontweight="bold")
apply_extent(ax2)

handles_b=[
    Patch(facecolor=c,label=typ.replace("_"," ").title())
    for typ,c in sorted(set(used_types))
]
handles_b += [Line2D([0],[0],color="black",lw=1.8,label="Shared State border")]
ax2.legend(handles=handles_b,loc="lower right",fontsize=5.8,title="Legend",title_fontsize=7,ncol=2)

# Panel C — accessibility context with labels, roads and bridges
for st,grp in corridors_map_p.groupby("state"):
    grp.plot(ax=ax3,facecolor=STATE_COLORS.get(st,"#eeeeee"),edgecolor="none",alpha=.28,zorder=1)

border_map_p.plot(ax=ax3,color="black",linewidth=1.8,zorder=8)
draw_drainage(ax3); draw_roads(ax3)

if len(bridges_p):
    bridges_p.plot(ax=ax3,color="#111111",linewidth=.65,zorder=9)

sett_map_p.plot(ax=ax3,facecolor="#fdb863",edgecolor="none",alpha=.72,zorder=3)

name_texts=[]
for _,r in sett_map_p.iterrows():
    nm=str(r.get("settlement_name","")).strip()
    if nm and nm.lower() not in ["nan","none"]:
        p=r.geometry.representative_point()
        name_texts.append(ax3.text(p.x,p.y,nm,fontsize=4.0,ha="left",va="bottom",zorder=12))

if len(name_texts) <= 120:
    adjust_text(name_texts,ax=ax3,expand=(1.05,1.12),force_text=(.3,.45),ensure_inside_axes=True,max_move=(12,12))

ax3.set_title("C. Border Settlements, Roads, Bridges and Drainage",fontsize=12,fontweight="bold")
apply_extent(ax3)

road_handles=[]
if len(roads_p) and "highway" in roads_p.columns:
    present=set(roads_p["highway"].dropna().astype(str))
    for hw in ["motorway","trunk","primary","secondary","tertiary","residential","unclassified","service"]:
        if hw in present:
            c,lw=ROAD_STYLE[hw]
            road_handles.append(Line2D([0],[0],color=c,lw=max(lw,.7),label=hw.title()))

drain_handles=[]
if len(drain_p) and "waterway" in drain_p.columns:
    present=set(drain_p["waterway"].dropna().astype(str))
    for w in ["river","stream","canal","drain"]:
        if w in present:
            c,lw,ls=DRAIN_STYLE[w]
            drain_handles.append(Line2D([0],[0],color=c,lw=max(lw,.7),ls=ls,label=w.title()))

handles_c=road_handles+drain_handles
if len(bridges_p):
    handles_c.append(Line2D([0],[0],color="#111111",lw=.8,label="Bridge"))
handles_c.append(Patch(facecolor="#fdb863",label="Border settlement"))

ax3.legend(handles=handles_c,loc="lower right",fontsize=5.6,title="Legend",title_fontsize=7,ncol=2)

# Panel D — two-State locator + summary
ax4.axis("off")
state1.to_crs(PAIR_CRS).plot(ax=ax4,facecolor=STATE_COLORS[STATE1],edgecolor="#555555",linewidth=.8)
state2.to_crs(PAIR_CRS).plot(ax=ax4,facecolor=STATE_COLORS[STATE2],edgecolor="#555555",linewidth=.8)
border_map_p.plot(ax=ax4,color="black",linewidth=2.2,zorder=8)

for st,g in [(STATE1,state1.to_crs(PAIR_CRS)),(STATE2,state2.to_crs(PAIR_CRS))]:
    p=g.geometry.iloc[0].representative_point()
    ax4.text(p.x,p.y,st,fontsize=9,fontweight="bold",ha="center",va="center")

summary_lines=[
    f"Interstate border: {STATE1} – {STATE2}",
    f"Shared border length: {shared_border.length/1000:,.1f} km",
    "Analysis corridor: 5 km inside each State/FCT",
    f"Settlement components: {len(settlement_csv):,}",
    f"Road features: {len(OSM_ROADS):,}",
    f"Bridge features: {len(OSM_BRIDGES):,}",
    f"Health facilities: {len(health):,}",
    f"Drainage features: {len(DRAINAGE):,}",
    "",
]
for _,r in summary_csv.iterrows():
    summary_lines.append(
        f"{r['corridor_state']}: {int(r['settlement_components']):,} settlements; "
        f"mean border distance {r['mean_border_distance_km']:.2f} km"
    )

ax4.text(
    .03,.04,"\n".join(summary_lines),
    transform=ax4.transAxes,
    va="bottom",ha="left",fontsize=8.5,
    bbox=dict(boxstyle="round,pad=.6",facecolor="#fafafa",edgecolor="#999999")
)

ax4.set_title("D. State-Pair Locator and Corridor Summary",fontsize=12,fontweight="bold")

fig.suptitle(
    f"Settlements within 5 km of the Shared {STATE1}–{STATE2} Border",
    fontsize=17,fontweight="bold",y=.97
)
fig.tight_layout(rect=[0.03,0.03,0.98,0.94])

multi_png=EXPORT/"01_interstate_border_multipanel_map.png"
fig.savefig(multi_png,dpi=300,bbox_inches="tight",facecolor="white")
plt.close(fig)

# -------------------------------------------------------
# STANDALONE SETTLEMENT MAP
# -------------------------------------------------------
fig,ax=plt.subplots(figsize=(13,10))
for st,grp in corridors_map_p.groupby("state"):
    grp.plot(ax=ax,facecolor=STATE_COLORS.get(st,"#eeeeee"),edgecolor="none",alpha=.45,zorder=1)
border_map_p.plot(ax=ax,color="black",linewidth=2,zorder=8)
sett_map_p.plot(ax=ax,facecolor="#fdb863",edgecolor="none",alpha=.82,zorder=3)
draw_drainage(ax); draw_roads(ax)
if len(bridges_p):
    bridges_p.plot(ax=ax,color="#111111",linewidth=.65,zorder=9)
apply_extent(ax)
ax.set_title(
    f"Settlements within 5 km of the {STATE1}–{STATE2} Shared Border",
    fontsize=14,fontweight="bold"
)
standalone_png=EXPORT/"02_interstate_border_settlements_map.png"
fig.savefig(standalone_png,dpi=300,bbox_inches="tight",facecolor="white")
plt.close(fig)

# ZIP CSV + PNG only.
zip_path = STUDENT_WORK_DIR/f"{safe(STATE1)}_{safe(STATE2)}_interstate_border_5km_outputs.zip"
if zip_path.exists():
    zip_path.unlink()

files_to_zip=[
    p.name for p in EXPORT.iterdir()
    if p.suffix.lower() in {".csv",".png"}
]

subprocess.run(
    ["zip","-q",str(zip_path),*files_to_zip],
    cwd=EXPORT,
    check=True
)

print("Outputs saved:")
for p in sorted(EXPORT.iterdir()):
    print(" -",p.name)
print("ZIP:",zip_path)

from google.colab import files
files.download(str(zip_path))
